# ML / LLM Whole-Solution Compute Graph

This notebook is the Jupyter version of the Python compute planner.

It models an ML/LLM production system as a **directed graph of nodes and edges** and calculates:

- data / ingestion compute
- training compute
- evaluation compute
- inference / serving compute
- observability overhead
- security overhead
- governance and policy overhead
- storage and network overhead
- dev / staging / production capacity
- Kubernetes replica and HPA requirements

The key idea is:

```text
requirements
    ↓
nodes + edges
    ↓
compute equations
    ↓
whole-solution capacity
    ↓
vendor-neutral infrastructure quantities
```

Cloud pricing comes **after** this calculation.


## 1. Imports

`networkx` represents the architecture DAG.

`pydantic` gives us typed production-style configuration models.

The notebook keeps everything visible rather than hiding logic behind a package.


In [ ]:
import math
import json
from dataclasses import dataclass, asdict
from typing import Dict, Optional, Any

import networkx as nx
from pydantic import BaseModel, Field


## 2. Configuration models

These models define the inputs to the compute problem.

The system intentionally separates:

1. the workload itself,
2. traffic/SLO requirements,
3. scheduled data/training/evaluation compute,
4. production platform overhead,
5. environment-specific scaling rules.


In [ ]:
class Workload(BaseModel):
    name: str
    model_type: str = "ml"
    model_memory_gb: float = 4.0
    cpu_per_replica: float = 2.0
    memory_per_replica_gb: float = 8.0
    gpu_per_replica: float = 0.0
    accelerator: Optional[str] = None


class Traffic(BaseModel):
    peak_rps: float = 10.0
    avg_request_seconds: float = 0.2


class SLO(BaseModel):
    p95_latency_ms: float = 1000
    availability: float = 0.99
    max_utilization: float = 0.70


class Serving(BaseModel):
    max_concurrency_per_replica: float = 8
    batching_efficiency: float = 1.0


class DataCompute(BaseModel):
    ingestion_cpu_cores: float = 0.0
    ingestion_memory_gb: float = 0.0
    preprocessing_cpu_core_hours_per_day: float = 0.0
    embedding_gpu_hours_per_day: float = 0.0
    storage_gb: float = 0.0


class TrainingCompute(BaseModel):
    enabled: bool = False
    gpu_count: float = 0.0
    gpu_hours_per_run: float = 0.0
    cpu_cores: float = 0.0
    cpu_hours_per_run: float = 0.0
    runs_per_day: float = 0.0
    checkpoint_storage_gb_per_run: float = 0.0


class EvaluationCompute(BaseModel):
    enabled: bool = True
    gpu_hours_per_run: float = 0.0
    cpu_core_hours_per_run: float = 1.0
    runs_per_day: float = 1.0
    artifact_storage_gb_per_run: float = 0.1


class OverheadComponent(BaseModel):
    enabled: bool = True
    cpu_percent: float = 0.0
    memory_percent: float = 0.0
    network_percent: float = 0.0
    request_cost_percent: float = 0.0
    storage_gb_per_replica_day: float = 0.0
    trace_sample_rate: float = 0.0


class PlatformOverhead(BaseModel):
    observability: OverheadComponent = Field(default_factory=OverheadComponent)
    security: OverheadComponent = Field(default_factory=OverheadComponent)
    governance: OverheadComponent = Field(default_factory=OverheadComponent)
    policy: OverheadComponent = Field(default_factory=OverheadComponent)
    platform: OverheadComponent = Field(default_factory=OverheadComponent)


class Environment(BaseModel):
    traffic_multiplier: float = 1.0
    training_multiplier: float = 1.0
    evaluation_multiplier: float = 1.0
    safety_factor: float = 1.2
    min_replicas: int = 1
    failure_reserve: int = 0
    overhead_multiplier: float = 1.0
    hpa_max_multiplier: float = 2.0


class PlannerConfig(BaseModel):
    workload: Workload
    traffic: Traffic
    slo: SLO
    serving: Serving
    data_compute: DataCompute = Field(default_factory=DataCompute)
    training_compute: TrainingCompute = Field(default_factory=TrainingCompute)
    evaluation_compute: EvaluationCompute = Field(default_factory=EvaluationCompute)
    platform_overhead: PlatformOverhead = Field(default_factory=PlatformOverhead)
    environments: Dict[str, Environment]


## 3. Example production workload

This example uses an LLM-oriented serving pattern with GPU-backed replicas.

The important part is that **the same workload definition can generate dev, staging, and prod**.


In [ ]:
config_dict = {
    "workload": {
        "name": "fraud-llm-router",
        "model_type": "llm",
        "model_memory_gb": 18,
        "cpu_per_replica": 4,
        "memory_per_replica_gb": 24,
        "gpu_per_replica": 1,
        "accelerator": "nvidia-l4",
    },

    "traffic": {
        "peak_rps": 24,
        "avg_request_seconds": 0.80,
    },

    "slo": {
        "p95_latency_ms": 900,
        "availability": 0.999,
        "max_utilization": 0.70,
    },

    "serving": {
        "max_concurrency_per_replica": 8,
        "batching_efficiency": 1.20,
    },

    "data_compute": {
        "ingestion_cpu_cores": 2,
        "ingestion_memory_gb": 4,
        "preprocessing_cpu_core_hours_per_day": 12,
        "embedding_gpu_hours_per_day": 1,
        "storage_gb": 500,
    },

    "training_compute": {
        "enabled": True,
        "gpu_count": 2,
        "gpu_hours_per_run": 4,
        "cpu_cores": 8,
        "cpu_hours_per_run": 2,
        "runs_per_day": 1,
        "checkpoint_storage_gb_per_run": 20,
    },

    "evaluation_compute": {
        "enabled": True,
        "gpu_hours_per_run": 1,
        "cpu_core_hours_per_run": 4,
        "runs_per_day": 2,
        "artifact_storage_gb_per_run": 1,
    },

    "platform_overhead": {
        "observability": {
            "enabled": True,
            "cpu_percent": 0.08,
            "memory_percent": 0.06,
            "network_percent": 0.06,
            "storage_gb_per_replica_day": 1.5,
            "request_cost_percent": 0.02,
            "trace_sample_rate": 0.10,
        },
        "security": {
            "enabled": True,
            "cpu_percent": 0.05,
            "memory_percent": 0.04,
            "network_percent": 0.03,
            "request_cost_percent": 0.01,
        },
        "governance": {
            "enabled": True,
            "cpu_percent": 0.03,
            "memory_percent": 0.03,
            "storage_gb_per_replica_day": 0.5,
            "request_cost_percent": 0.02,
        },
        "policy": {
            "enabled": True,
            "cpu_percent": 0.02,
            "memory_percent": 0.01,
            "request_cost_percent": 0.015,
        },
        "platform": {
            "enabled": True,
            "cpu_percent": 0.03,
            "memory_percent": 0.03,
            "network_percent": 0.02,
        },
    },

    "environments": {
        "dev": {
            "traffic_multiplier": 0.10,
            "training_multiplier": 0.25,
            "evaluation_multiplier": 0.50,
            "safety_factor": 1.10,
            "min_replicas": 1,
            "failure_reserve": 0,
            "overhead_multiplier": 0.65,
            "hpa_max_multiplier": 2.0,
        },
        "staging": {
            "traffic_multiplier": 0.50,
            "training_multiplier": 0.50,
            "evaluation_multiplier": 1.00,
            "safety_factor": 1.30,
            "min_replicas": 2,
            "failure_reserve": 1,
            "overhead_multiplier": 0.90,
            "hpa_max_multiplier": 2.5,
        },
        "prod": {
            "traffic_multiplier": 1.00,
            "training_multiplier": 1.00,
            "evaluation_multiplier": 1.00,
            "safety_factor": 1.60,
            "min_replicas": 3,
            "failure_reserve": 1,
            "overhead_multiplier": 1.00,
            "hpa_max_multiplier": 3.0,
        },
    },
}

cfg = PlannerConfig.model_validate(config_dict)
cfg


## 4. Platform-overhead result type

Observability, security, governance, policy, and shared platform services all consume capacity.

Instead of pretending these are free, we model them as separate compute taxes.


In [ ]:
@dataclass
class ComponentTax:
    cpu: float = 0.0
    memory_gb: float = 0.0
    network_units: float = 0.0
    storage_gb_day: float = 0.0
    request_equivalent_rps: float = 0.0


## 5. Compute overhead for one graph node

This function calculates the tax introduced by one platform capability.

Examples:

- observability
- security
- governance
- policy-as-code
- Kubernetes/platform services

The coefficients are placeholders that should eventually be calibrated from real telemetry.


In [ ]:
def component_tax(
    component: OverheadComponent,
    base_cpu: float,
    base_memory: float,
    replicas: int,
    rps: float,
    env_multiplier: float,
) -> ComponentTax:

    if not component.enabled:
        return ComponentTax()

    cpu = base_cpu * component.cpu_percent * env_multiplier
    memory = base_memory * component.memory_percent * env_multiplier

    # Traffic-sensitive request overhead.
    request_equivalent_rps = (
        rps
        * component.request_cost_percent
        * env_multiplier
    )

    # Generic normalized request-to-CPU conversion.
    # Later this should be replaced with empirical measurements.
    cpu += request_equivalent_rps * 0.01

    network_units = (
        rps * component.network_percent * env_multiplier
        + rps * component.trace_sample_rate * 0.01 * env_multiplier
    )

    storage_gb_day = (
        replicas
        * component.storage_gb_per_replica_day
        * env_multiplier
    )

    return ComponentTax(
        cpu=cpu,
        memory_gb=memory,
        network_units=network_units,
        storage_gb_day=storage_gb_day,
        request_equivalent_rps=request_equivalent_rps,
    )


## 6. Build the ML/LLM architecture graph

The graph is the architectural model.

Nodes are not merely documentation labels. They represent either:

- requirements,
- compute-producing workloads,
- constraints,
- derived capacity,
- or overhead.

Edges express dependencies and scaling relationships.


In [ ]:
def build_graph(cfg: PlannerConfig, env_name: str) -> nx.DiGraph:
    g = nx.DiGraph()

    nodes = [
        ("business_decision", {"kind": "requirement"}),
        ("evidence", {"kind": "requirement"}),
        ("slo", {"kind": "constraint"}),

        ("data", {"kind": "compute"}),
        ("ingestion", {"kind": "compute"}),
        ("features_embeddings", {"kind": "compute"}),

        ("training", {"kind": "compute"}),
        ("evaluation", {"kind": "compute"}),

        ("traffic", {"kind": "load"}),
        ("latency", {"kind": "constraint"}),
        ("availability", {"kind": "constraint"}),
        ("concurrency", {"kind": "derived"}),

        ("model_runtime", {"kind": "compute"}),
        ("serving_replicas", {"kind": "derived"}),
        ("feedback", {"kind": "compute"}),

        ("observability", {"kind": "overhead"}),
        ("security", {"kind": "overhead"}),
        ("governance", {"kind": "overhead"}),
        ("policy", {"kind": "overhead"}),
        ("platform", {"kind": "overhead"}),

        ("effective_footprint", {"kind": "derived"}),
        ("autoscaling", {"kind": "deployment"}),
        ("kubernetes", {"kind": "deployment"}),
    ]

    g.add_nodes_from(nodes)

    edges = [
        ("business_decision", "evidence"),
        ("evidence", "slo"),

        ("slo", "latency"),
        ("slo", "availability"),

        ("data", "ingestion"),
        ("ingestion", "features_embeddings"),
        ("features_embeddings", "training"),
        ("training", "evaluation"),
        ("evaluation", "model_runtime"),

        ("traffic", "concurrency"),
        ("latency", "concurrency"),

        ("concurrency", "serving_replicas"),
        ("availability", "serving_replicas"),
        ("model_runtime", "serving_replicas"),

        ("serving_replicas", "feedback"),
        ("feedback", "evaluation"),

        ("data", "observability"),
        ("training", "observability"),
        ("evaluation", "observability"),
        ("serving_replicas", "observability"),

        ("training", "security"),
        ("serving_replicas", "security"),

        ("training", "governance"),
        ("evaluation", "governance"),
        ("serving_replicas", "governance"),

        ("governance", "policy"),
        ("serving_replicas", "platform"),

        ("observability", "effective_footprint"),
        ("security", "effective_footprint"),
        ("governance", "effective_footprint"),
        ("policy", "effective_footprint"),
        ("platform", "effective_footprint"),

        ("ingestion", "effective_footprint"),
        ("features_embeddings", "effective_footprint"),
        ("training", "effective_footprint"),
        ("evaluation", "effective_footprint"),
        ("serving_replicas", "effective_footprint"),

        ("effective_footprint", "autoscaling"),
        ("autoscaling", "kubernetes"),
    ]

    g.add_edges_from(edges)

    return g


## 7. Inspect the graph

This prints every node and edge.

That makes the architecture fully inspectable and machine-readable.


In [ ]:
graph = build_graph(cfg, "prod")

print("NODES")
for node, attrs in graph.nodes(data=True):
    print(f"{node:24} {attrs}")

print("\nEDGES")
for source, target in graph.edges:
    print(f"{source:24} -> {target}")


## 8. Serving capacity math

The serving calculation starts with traffic.

A useful approximation is Little's Law:

```text
concurrency ≈ arrival_rate × average_service_time
```

Then we divide that workload across replicas, while preserving utilization headroom and batching efficiency.


In [ ]:
def calculate_serving(cfg: PlannerConfig, env_name: str):
    env = cfg.environments[env_name]

    rps = cfg.traffic.peak_rps * env.traffic_multiplier

    estimated_concurrency = (
        rps
        * cfg.traffic.avg_request_seconds
    )

    effective_capacity_per_replica = (
        cfg.serving.max_concurrency_per_replica
        * cfg.slo.max_utilization
        * cfg.serving.batching_efficiency
    )

    demand_replicas = math.ceil(
        (estimated_concurrency * env.safety_factor)
        / max(effective_capacity_per_replica, 0.01)
    )

    raw_replicas = max(
        env.min_replicas,
        demand_replicas,
    )

    recommended_replicas = (
        raw_replicas
        + env.failure_reserve
    )

    return {
        "rps": rps,
        "estimated_concurrency": estimated_concurrency,
        "effective_capacity_per_replica": effective_capacity_per_replica,
        "demand_replicas": demand_replicas,
        "raw_replicas": raw_replicas,
        "recommended_replicas": recommended_replicas,
    }


calculate_serving(cfg, "prod")


## 9. Whole-environment compute solver

This is the central calculation.

For one environment it computes:

- serving load
- data pipeline compute
- training compute
- evaluation compute
- production overhead
- total steady-state footprint
- scheduled GPU workload
- storage growth
- autoscaling range


In [ ]:
def plan_environment(
    cfg: PlannerConfig,
    env_name: str,
) -> Dict[str, Any]:

    env = cfg.environments[env_name]

    # --------------------------------------------------
    # SERVING
    # --------------------------------------------------

    rps = (
        cfg.traffic.peak_rps
        * env.traffic_multiplier
    )

    raw_concurrency = (
        rps
        * cfg.traffic.avg_request_seconds
    )

    effective_capacity_per_replica = (
        cfg.serving.max_concurrency_per_replica
        * cfg.slo.max_utilization
        * cfg.serving.batching_efficiency
    )

    demand_replicas = math.ceil(
        (raw_concurrency * env.safety_factor)
        / max(effective_capacity_per_replica, 0.01)
    )

    raw_serving_replicas = max(
        env.min_replicas,
        demand_replicas,
    )

    recommended_replicas = (
        raw_serving_replicas
        + env.failure_reserve
    )

    serving_cpu = (
        recommended_replicas
        * cfg.workload.cpu_per_replica
    )

    serving_memory = (
        recommended_replicas
        * cfg.workload.memory_per_replica_gb
    )

    serving_gpu = (
        recommended_replicas
        * cfg.workload.gpu_per_replica
    )

    # --------------------------------------------------
    # DATA
    # --------------------------------------------------

    data_cpu = (
        cfg.data_compute.ingestion_cpu_cores
        + cfg.data_compute.preprocessing_cpu_core_hours_per_day / 24.0
    )

    data_memory = (
        cfg.data_compute.ingestion_memory_gb
    )

    data_gpu_hours_day = (
        cfg.data_compute.embedding_gpu_hours_per_day
    )

    data_storage = (
        cfg.data_compute.storage_gb
    )

    # --------------------------------------------------
    # TRAINING
    # --------------------------------------------------

    if cfg.training_compute.enabled:
        training_runs = (
            cfg.training_compute.runs_per_day
            * env.training_multiplier
        )
    else:
        training_runs = 0.0

    training_gpu_hours_day = (
        cfg.training_compute.gpu_count
        * cfg.training_compute.gpu_hours_per_run
        * training_runs
    )

    training_cpu_core_hours_day = (
        cfg.training_compute.cpu_cores
        * cfg.training_compute.cpu_hours_per_run
        * training_runs
    )

    training_storage_day = (
        cfg.training_compute.checkpoint_storage_gb_per_run
        * training_runs
    )

    # --------------------------------------------------
    # EVALUATION
    # --------------------------------------------------

    if cfg.evaluation_compute.enabled:
        evaluation_runs = (
            cfg.evaluation_compute.runs_per_day
            * env.evaluation_multiplier
        )
    else:
        evaluation_runs = 0.0

    evaluation_gpu_hours_day = (
        cfg.evaluation_compute.gpu_hours_per_run
        * evaluation_runs
    )

    evaluation_cpu_core_hours_day = (
        cfg.evaluation_compute.cpu_core_hours_per_run
        * evaluation_runs
    )

    evaluation_storage_day = (
        cfg.evaluation_compute.artifact_storage_gb_per_run
        * evaluation_runs
    )

    # Scheduled CPU work represented as average concurrent CPU.
    scheduled_cpu_avg = (
        training_cpu_core_hours_day
        + evaluation_cpu_core_hours_day
    ) / 24.0

    # --------------------------------------------------
    # BASE FOOTPRINT BEFORE PLATFORM TAX
    # --------------------------------------------------

    overhead_base_cpu = (
        serving_cpu
        + data_cpu
        + scheduled_cpu_avg
    )

    overhead_base_memory = (
        serving_memory
        + data_memory
    )

    # --------------------------------------------------
    # PLATFORM / SECURITY / OBSERVABILITY TAX
    # --------------------------------------------------

    taxes = {}

    for name in [
        "observability",
        "security",
        "governance",
        "policy",
        "platform",
    ]:
        component = getattr(
            cfg.platform_overhead,
            name,
        )

        taxes[name] = component_tax(
            component,
            overhead_base_cpu,
            overhead_base_memory,
            recommended_replicas,
            rps,
            env.overhead_multiplier,
        )

    overhead_cpu = sum(
        x.cpu
        for x in taxes.values()
    )

    overhead_memory = sum(
        x.memory_gb
        for x in taxes.values()
    )

    overhead_network = sum(
        x.network_units
        for x in taxes.values()
    )

    overhead_storage_day = sum(
        x.storage_gb_day
        for x in taxes.values()
    )

    # --------------------------------------------------
    # EFFECTIVE WHOLE-SOLUTION FOOTPRINT
    # --------------------------------------------------

    effective_cpu = (
        overhead_base_cpu
        + overhead_cpu
    )

    effective_memory = (
        overhead_base_memory
        + overhead_memory
    )

    total_daily_gpu_hours = (
        training_gpu_hours_day
        + evaluation_gpu_hours_day
        + data_gpu_hours_day
    )

    total_storage = (
        data_storage
        + training_storage_day
        + evaluation_storage_day
        + overhead_storage_day
    )

    replica_equivalent_overhead = (
        overhead_cpu
        / cfg.workload.cpu_per_replica
        if cfg.workload.cpu_per_replica > 0
        else 0.0
    )

    # --------------------------------------------------
    # KUBERNETES AUTOSCALING RANGE
    # --------------------------------------------------

    hpa_min = recommended_replicas

    hpa_max = max(
        hpa_min + 1,
        math.ceil(
            recommended_replicas
            * env.hpa_max_multiplier
        ),
    )

    graph = build_graph(
        cfg,
        env_name,
    )

    return {
        "environment": env_name,

        "traffic": {
            "target_rps": round(rps, 3),
            "estimated_concurrency": round(raw_concurrency, 3),
        },

        "data_compute": {
            "avg_cpu_cores": round(data_cpu, 3),
            "memory_gb": round(data_memory, 3),
            "embedding_gpu_hours_per_day": round(data_gpu_hours_day, 3),
            "storage_gb": round(data_storage, 3),
        },

        "training_compute": {
            "runs_per_day": round(training_runs, 3),
            "gpu_hours_per_day": round(training_gpu_hours_day, 3),
            "cpu_core_hours_per_day": round(training_cpu_core_hours_day, 3),
            "checkpoint_storage_gb_per_day": round(training_storage_day, 3),
        },

        "evaluation_compute": {
            "runs_per_day": round(evaluation_runs, 3),
            "gpu_hours_per_day": round(evaluation_gpu_hours_day, 3),
            "cpu_core_hours_per_day": round(evaluation_cpu_core_hours_day, 3),
            "artifact_storage_gb_per_day": round(evaluation_storage_day, 3),
        },

        "serving_compute": {
            "demand_replicas": demand_replicas,
            "raw_serving_replicas": raw_serving_replicas,
            "failure_reserve": env.failure_reserve,
            "recommended_replicas": recommended_replicas,
            "hpa_min": hpa_min,
            "hpa_max": hpa_max,
            "cpu_cores": round(serving_cpu, 3),
            "memory_gb": round(serving_memory, 3),
            "gpus": round(serving_gpu, 3),
            "accelerator": cfg.workload.accelerator,
        },

        "platform_overhead": {
            name: {
                k: round(v, 4)
                for k, v in asdict(tax).items()
            }
            for name, tax in taxes.items()
        },

        "effective_compute": {
            "steady_state_cpu_cores": round(effective_cpu, 3),
            "steady_state_memory_gb": round(effective_memory, 3),
            "serving_gpus": round(serving_gpu, 3),
            "scheduled_gpu_hours_per_day": round(total_daily_gpu_hours, 3),
            "storage_gb_plus_daily_growth": round(total_storage, 3),
            "network_units": round(overhead_network, 3),
            "overhead_cpu_cores": round(overhead_cpu, 3),
            "replica_equivalent_overhead": round(
                replica_equivalent_overhead,
                3,
            ),
        },

        "graph": {
            "nodes": list(graph.nodes),
            "edges": [
                list(edge)
                for edge in graph.edges
            ],
        },
    }


## 10. Solve production

This gives the full calculated production footprint.


In [ ]:
prod_plan = plan_environment(
    cfg,
    "prod",
)

print(
    json.dumps(
        prod_plan,
        indent=2,
    )
)


## 11. Solve dev, staging, and production

This runs the same compute graph against all environments.


In [ ]:
def plan_all(
    cfg: PlannerConfig,
) -> Dict[str, Any]:

    return {
        "workload": cfg.workload.name,
        "environments": {
            env_name: plan_environment(
                cfg,
                env_name,
            )
            for env_name in cfg.environments
        },
    }


full_plan = plan_all(cfg)

print(
    json.dumps(
        full_plan,
        indent=2,
    )
)


## 12. Compact comparison table

This gives the executive view of the environment calculation.


In [ ]:
import pandas as pd

rows = []

for env_name, plan in full_plan["environments"].items():

    rows.append({
        "environment": env_name,
        "replicas": plan["serving_compute"]["recommended_replicas"],
        "hpa_min": plan["serving_compute"]["hpa_min"],
        "hpa_max": plan["serving_compute"]["hpa_max"],
        "serving_gpus": plan["serving_compute"]["gpus"],
        "steady_cpu": plan["effective_compute"]["steady_state_cpu_cores"],
        "steady_ram_gb": plan["effective_compute"]["steady_state_memory_gb"],
        "scheduled_gpu_hours_day": plan["effective_compute"]["scheduled_gpu_hours_per_day"],
        "overhead_cpu": plan["effective_compute"]["overhead_cpu_cores"],
        "storage_plus_growth_gb": plan["effective_compute"]["storage_gb_plus_daily_growth"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df


## 13. Inspect security and observability separately

These are intentionally first-class compute consumers.

This lets the system show how much production capacity is required **because** you operate the workload properly.


In [ ]:
prod_overhead = full_plan["environments"]["prod"]["platform_overhead"]

overhead_df = pd.DataFrame([
    {
        "component": name,
        "cpu": values["cpu"],
        "memory_gb": values["memory_gb"],
        "network_units": values["network_units"],
        "storage_gb_day": values["storage_gb_day"],
        "request_equivalent_rps": values["request_equivalent_rps"],
    }
    for name, values in prod_overhead.items()
])

overhead_df


## 14. Show the architecture DAG visually

This uses NetworkX only for visualization.

The compute logic remains in the equations above.


In [ ]:
import matplotlib.pyplot as plt

prod_graph = build_graph(
    cfg,
    "prod",
)

plt.figure(figsize=(16, 12))

pos = nx.spring_layout(
    prod_graph,
    seed=7,
)

nx.draw_networkx(
    prod_graph,
    pos=pos,
    with_labels=True,
    node_size=2600,
    font_size=8,
    arrows=True,
)

plt.title(
    "ML / LLM Whole-Solution Compute Graph"
)

plt.axis("off")
plt.show()


## 15. The next architectural evolution: executable edges

Right now the DAG captures architectural dependencies while the formulas live in Python functions.

The stronger form is to make edges carry their own transform functions.

Conceptually:

```python
graph.add_edge(
    "traffic",
    "concurrency",
    transform=littles_law,
)

graph.add_edge(
    "concurrency",
    "serving_replicas",
    transform=replica_solver,
)

graph.add_edge(
    "serving_replicas",
    "observability",
    transform=telemetry_overhead,
)
```

That would turn the graph itself into the executable compute planner.


In [ ]:
def littles_law(rps: float, service_seconds: float) -> float:
    return rps * service_seconds


def replica_solver(
    concurrency: float,
    capacity_per_replica: float,
    safety_factor: float,
) -> int:

    return math.ceil(
        (concurrency * safety_factor)
        / max(capacity_per_replica, 0.01)
    )


example_concurrency = littles_law(
    rps=24,
    service_seconds=0.8,
)

example_replicas = replica_solver(
    concurrency=example_concurrency,
    capacity_per_replica=(
        8
        * 0.70
        * 1.20
    ),
    safety_factor=1.60,
)

print(
    "Concurrency:",
    example_concurrency,
)

print(
    "Demand replicas:",
    example_replicas,
)


## 16. Vendor-neutral compute specification

This is the point where the compute problem ends.

The resulting quantities can then be compared with:

- AWS
- Azure
- GCP
- private cloud
- bare metal
- colocation
- GPU appliances

The pricing engine should consume this output rather than recomputing architecture.


In [ ]:
vendor_neutral_spec = {
    env: {
        "cpu_cores": plan["effective_compute"]["steady_state_cpu_cores"],
        "memory_gb": plan["effective_compute"]["steady_state_memory_gb"],
        "serving_gpus": plan["effective_compute"]["serving_gpus"],
        "scheduled_gpu_hours_per_day": plan["effective_compute"]["scheduled_gpu_hours_per_day"],
        "storage_gb_plus_daily_growth": plan["effective_compute"]["storage_gb_plus_daily_growth"],
        "network_units": plan["effective_compute"]["network_units"],
        "hpa_min": plan["serving_compute"]["hpa_min"],
        "hpa_max": plan["serving_compute"]["hpa_max"],
    }
    for env, plan in full_plan["environments"].items()
}

print(
    json.dumps(
        vendor_neutral_spec,
        indent=2,
    )
)


# Summary

This notebook expresses the ML/LLM architecture as a compute problem:

```text
product requirement
      ↓
ML architecture graph
      ↓
data + training + evaluation + serving
      ↓
security + observability + governance + policy
      ↓
failure reserve + autoscaling
      ↓
whole-solution compute footprint
      ↓
dev / staging / production
      ↓
cloud or hardware price comparison
```

The most important architectural principle is:

> **Compute first. Price second.**

The graph should eventually become executable, with weighted edges carrying the actual transforms used by the solver.
